# Gold Layer Pipeline - Fraud Detection Analytics

**Purpose**: Create optimized, aggregated tables for dashboard and analytics

**Gold Tables Created**:
1. fraud_by_day_of_week – Which day(s) of the week sees the highest number of fraudulent transactions?
2. fraud_rate_trend – Fraud rate trend over past month
3. top_fraud_users – Users with largest number of flagged transactions
4. user_spending_anomalies – Users with sharp rise in transaction amount vs weekly average
5. fraud_by_merchant_category – Merchant categories with highest fraud rate
6. fraud_by_merchant – Specific merchants with unusually high fraud volume
7. fraud_by_time_of_day – Fraud distribution by time of day
8. fraud_vs_nonfraud_amount – Average transaction amount for fraud vs non-fraud
9. fraud_by_device_type – Device types most associated with fraudulent transactions
10. shared_device_fraud – Instances where same device used by multiple fraud users
11. daily_fraud_losses – Total monetary losses due to fraud each day
12. weekly_unique_fraud_users – Unique users committing fraudulent transactions per week
13. fraud_seasonal_patterns – Seasonal or monthly spikes in fraud
14. user_behavior_before_after_fraud – User behavior changes before vs after fraud
15. fraud_by_transaction_value – Fraud patterns by transaction value ranges


## Setup

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Configuration
SILVER_PATH = "databricks_workspace.silver"
GOLD_PATH = "databricks_workspace.gold"

# Load silver tables
df_trans = spark.table(f"{SILVER_PATH}.transactions")
df_cards = spark.table(f"{SILVER_PATH}.cards")
df_users = spark.table(f"{SILVER_PATH}.users")

## 1. Fraud by Day of Week

In [0]:
gold_fraud_by_dayofweek = (df_trans
    .groupBy("transaction_dayofweek", "transaction_dayname")
    .agg(
        count("*").alias("total_transactions"),
        sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
        sum(when(col("is_fraud") == False, 1).otherwise(0)).alias("legitimate_transactions"),
        (sum(when(col("is_fraud") == True, 1).otherwise(0)) / count("*") * 100).alias("fraud_rate_pct"),
        sum(when(col("is_fraud") == True, col("amount_clean")).otherwise(0)).alias("fraud_amount_total")
    )
    .orderBy("transaction_dayofweek")
)

gold_fraud_by_dayofweek.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.fraud_by_day_of_week")
print("Created: gold.fraud_by_day_of_week")
display(gold_fraud_by_dayofweek)

Created: gold.fraud_by_day_of_week


transaction_dayofweek,transaction_dayname,total_transactions,fraud_transactions,legitimate_transactions,fraud_rate_pct,fraud_amount_total
1,Sunday,1899044,2646,1270225,0.13933326452678296,327420.29
2,Monday,1896914,1747,1269414,0.09209695326198236,198255.49
3,Tuesday,1897678,2037,1268939,0.10734170918353904,228815.71
4,Wednesday,1895871,1102,1270075,0.05812631766612812,99237.05
5,Thursday,1918666,2082,1283019,0.1085128938543759,227773.35
6,Friday,1895372,2284,1267525,0.12050404880941579,247422.08
7,Saturday,1902370,1434,1272434,0.07537965800554046,140724.81


## 2. Fraud Rate Trend Over Past Month

In [0]:
# Calculate fraud rate by date
gold_fraud_rate_trend = (df_trans
    .groupBy("transaction_date")
    .agg(
        count("*").alias("total_transactions"),
        sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
        (sum(when(col("is_fraud") == True, 1).otherwise(0)) / count("*") * 100).alias("fraud_rate_pct"),
        sum(when(col("is_fraud") == True, col("amount_clean")).otherwise(0)).alias("fraud_amount"),
        sum(col("amount_clean")).alias("total_amount")
    )
    .orderBy("transaction_date")
    
    # Add 7-day moving average for smoothing
    .withColumn("fraud_rate_7day_avg", 
        avg("fraud_rate_pct").over(
            Window.orderBy("transaction_date")
            .rowsBetween(-6, 0)
        )
    )
)

gold_fraud_rate_trend.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.fraud_rate_trend")
print("Created: gold.fraud_rate_trend")
display(gold_fraud_rate_trend.orderBy(desc("transaction_date")).limit(30))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Created: gold.fraud_rate_trend


transaction_date,total_transactions,fraud_transactions,fraud_rate_pct,fraud_amount,total_amount,fraud_rate_7day_avg
2019-10-31,3978,0,0.0,0.00,158747.19,0.0594661166711019
2019-10-30,3562,0,0.0,0.00,157791.07,0.0891122163561121
2019-10-29,3552,0,0.0,0.00,160928.07,0.0891122163561121
2019-10-28,3944,0,0.0,0.00,168231.71,0.13417756110599943
2019-10-27,3866,1,0.02586652871184687,-210.00,154794.08,0.13417756110599943
2019-10-26,3813,6,0.15735641227380015,976.92,160283.91,0.13048234271859274
2019-10-25,3862,9,0.2330398757120663,132.97,156800.84,0.14424261751806652
2019-10-24,3855,8,0.20752269779507135,1165.86,164125.65,0.15069396268153534
2019-10-23,3668,0,0.0,0.00,155575.11,0.1612070393683261
2019-10-22,3487,11,0.31545741324921134,1225.82,160263.74,0.1612070393683261


## 3. Top Fraud Users

In [0]:
gold_top_fraud_users = (df_trans
    .filter(col("is_fraud") == True)
    .join(df_users, df_trans.client_id == df_users.user_id, "left")
    .groupBy(
        df_trans.client_id,
        df_users.current_age,
        df_users.gender,
        df_users.credit_score,
        df_users.yearly_income_clean,
        df_users.num_credit_cards
    )
    .agg(
        count("*").alias("fraud_transaction_count"),
        sum(col("amount_clean")).alias("total_fraud_amount"),
        avg(col("amount_clean")).alias("avg_fraud_amount"),
        min(col("transaction_date")).alias("first_fraud_date"),
        max(col("transaction_date")).alias("last_fraud_date"),
        countDistinct("card_id").alias("cards_used"),
        countDistinct("merchant_id").alias("merchants_involved")
    )
    .orderBy(desc("fraud_transaction_count"))
)

gold_top_fraud_users.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.top_fraud_users")
print("Created: gold.top_fraud_users")
display(gold_top_fraud_users.limit(20))

Created: gold.top_fraud_users


client_id,current_age,gender,credit_score,yearly_income_clean,num_credit_cards,fraud_transaction_count,total_fraud_amount,avg_fraud_amount,first_fraud_date,last_fraud_date,cards_used,merchants_involved
1102,49,Female,707,94733.00,5,58,7615.21,131.296724,2010-05-03,2018-03-15,3,43
209,61,Female,716,29206.00,6,52,4855.87,93.382115,2010-07-02,2019-10-25,4,36
27,78,Female,613,23821.00,8,45,2932.16,65.159111,2012-08-07,2019-01-26,5,32
155,86,Female,688,9678.00,6,44,3536.50,80.375000,2010-01-26,2019-05-03,4,33
1128,47,Male,680,34606.00,5,43,4699.15,109.282558,2010-11-07,2019-08-31,5,26
1851,48,Male,727,75682.00,5,42,6102.57,145.299286,2010-05-21,2018-09-21,4,34
989,78,Male,727,113514.00,8,42,7499.97,178.570714,2012-03-14,2016-12-13,4,25
1741,92,Male,707,24960.00,9,42,3575.72,85.136190,2010-03-22,2019-03-25,4,34
1649,64,Male,729,399.00,3,41,3585.29,87.446098,2010-05-06,2019-08-15,2,34
359,66,Male,799,57383.00,4,39,3677.28,94.289231,2014-01-26,2018-11-22,3,27


## 4. User Spending Anomalies (Sharp Rise vs Weekly Average)

In [0]:
# Calculate weekly averages and current week spending
user_weekly_spending = (df_trans
    .groupBy("client_id", "transaction_week", "transaction_year")
    .agg(
        count("*").alias("weekly_transactions"),
        sum("amount_clean").alias("weekly_total_amount"),
        avg("amount_clean").alias("weekly_avg_amount")
    )
)

# Calculate overall user averages
user_avg_spending = (user_weekly_spending
    .groupBy("client_id")
    .agg(
        avg("weekly_total_amount").alias("user_avg_weekly_total"),
        avg("weekly_avg_amount").alias("user_avg_transaction_amount"),
        stddev("weekly_total_amount").alias("weekly_total_stddev")
    )
)

# Find current week and compare
current_week = df_trans.select(max("transaction_week")).first()[0]
current_year = df_trans.select(max("transaction_year")).first()[0]

# Use aliases to avoid ambiguous column references
weekly_alias = user_weekly_spending.alias("weekly")
avg_alias = user_avg_spending.alias("avg")
users_alias = df_users.alias("users")

gold_spending_anomalies = (weekly_alias
    .filter((col("weekly.transaction_week") == current_week) & (col("weekly.transaction_year") == current_year))
    .join(avg_alias, col("weekly.client_id") == col("avg.client_id"), "inner")
    .join(users_alias, col("weekly.client_id") == col("users.user_id"), "left")
    .withColumn("spending_increase_pct", 
        ((col("weekly.weekly_total_amount") - col("avg.user_avg_weekly_total")) / col("avg.user_avg_weekly_total") * 100)
    )
    .withColumn("z_score",
        when(col("avg.weekly_total_stddev") > 0,
             (col("weekly.weekly_total_amount") - col("avg.user_avg_weekly_total")) / col("avg.weekly_total_stddev")
        ).otherwise(0)
    )
    .filter(col("spending_increase_pct") > 50)  # At least 50% increase
    .select(
        col("weekly.client_id"),
        col("users.current_age"),
        col("users.credit_score"),
        col("weekly.weekly_total_amount"),
        col("avg.user_avg_weekly_total"),
        col("spending_increase_pct"),
        col("z_score"),
        col("weekly.weekly_transactions")
    )
    .orderBy(desc("spending_increase_pct"))
)

gold_spending_anomalies.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.user_spending_anomalies")
print("Created: gold.user_spending_anomalies")
display(gold_spending_anomalies.limit(20))

Created: gold.user_spending_anomalies


client_id,current_age,credit_score,weekly_total_amount,user_avg_weekly_total,spending_increase_pct,z_score,weekly_transactions


## 5. Fraud by Merchant Category

In [0]:
gold_fraud_by_category = (df_trans
    .groupBy("mcc_code", "mcc_description")
    .agg(
        count("*").alias("total_transactions"),
        sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
        (sum(when(col("is_fraud") == True, 1).otherwise(0)) / count("*") * 100).alias("fraud_rate_pct"),
        sum(when(col("is_fraud") == True, col("amount_clean")).otherwise(0)).alias("fraud_amount_total"),
        avg(when(col("is_fraud") == True, col("amount_clean"))).alias("avg_fraud_amount"),
        countDistinct(when(col("is_fraud") == True, col("client_id"))).alias("unique_fraud_users")
    )
    .filter(col("total_transactions") >= 10)  # Minimum threshold for meaningful rate
    .orderBy(desc("fraud_rate_pct"))
)

gold_fraud_by_category.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.fraud_by_merchant_category")
print("Created: gold.fraud_by_merchant_category")
display(gold_fraud_by_category.limit(20))

Created: gold.fraud_by_merchant_category


mcc_code,mcc_description,total_transactions,fraud_transactions,fraud_rate_pct,fraud_amount_total,avg_fraud_amount,unique_fraud_users
4411,Cruise Lines,428,165,38.55140186915888,185946.78,1126.950182,144
5733,Music Stores - Musical Instruments,319,76,23.824451410658305,7562.26,99.503421,72
3006,Miscellaneous Fabricated Metal Products,351,29,8.262108262108262,8543.36,294.598621,29
5045,"Computers, Computer Peripheral Equipment",2793,204,7.303974221267455,25615.11,125.564265,180
3144,Floor Covering Stores,334,23,6.88622754491018,6643.92,288.866087,23
5732,Electronics Stores,6997,402,5.74531942260969,61171.38,152.167612,313
3005,Miscellaneous Metal Fabrication,391,22,5.626598465473146,6943.78,315.626364,22
3009,Fabricated Structural Metal Products,408,22,5.392156862745098,6234.74,283.397273,22
5094,Precious Stones and Metals,5180,242,4.671814671814672,26575.96,109.818017,206
3007,Coated and Laminated Products,381,17,4.4619422572178475,5642.57,331.915882,17


## 6. Fraud by Specific Merchant

In [0]:
gold_fraud_by_merchant = (df_trans
    .groupBy(
        "merchant_id",
        "merchant_city",
        "merchant_state",
        "mcc_description"
    )
    .agg(
        count("*").alias("total_transactions"),
        sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
        (sum(when(col("is_fraud") == True, 1).otherwise(0)) / count("*") * 100).alias("fraud_rate_pct"),
        sum(when(col("is_fraud") == True, col("amount_clean")).otherwise(0)).alias("fraud_amount_total"),
        countDistinct(when(col("is_fraud") == True, col("client_id"))).alias("unique_fraud_users")
    )
    .filter(col("total_transactions") >= 5)
    .orderBy(desc("fraud_transactions"))
)

gold_fraud_by_merchant.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.fraud_by_merchant")
print("Created: gold.fraud_by_merchant")
display(gold_fraud_by_merchant.limit(20))

Created: gold.fraud_by_merchant


merchant_id,merchant_city,merchant_state,mcc_description,total_transactions,fraud_transactions,fraud_rate_pct,fraud_amount_total,unique_fraud_users
60569,ONLINE,null,Wholesale Clubs,1104,759,68.75,82309.67,483
27092,ONLINE,null,Money Transfer,1055,715,67.77251184834124,64747.02,440
76639,ONLINE,null,Electronics Stores,397,284,71.53652392947103,43344.88,235
32858,ONLINE,null,Department Stores,431,283,65.66125290023201,27824.99,228
83018,Rome,Italy,Discount Stores,391,236,60.35805626598465,13363.74,173
48919,Rome,Italy,Department Stores,332,221,66.56626506024097,18811.81,167
99370,Rome,Italy,Department Stores,326,195,59.8159509202454,12686.28,153
47399,ONLINE,null,"Digital Goods - Media, Books, Apps",20858,176,0.8438009396874101,8749.92,158
34490,ONLINE,null,Miscellaneous Home Furnishing Stores,230,160,69.56521739130434,17468.52,146
88260,Rome,Italy,"Grocery Stores, Supermarkets",390,149,38.205128205128204,5140.36,126


## 7. Fraud by Time of Day

In [0]:
gold_fraud_by_time = (df_trans
    .groupBy("time_of_day", "transaction_hour")
    .agg(
        count("*").alias("total_transactions"),
        sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
        (sum(when(col("is_fraud") == True, 1).otherwise(0)) / count("*") * 100).alias("fraud_rate_pct"),
        sum(when(col("is_fraud") == True, col("amount_clean")).otherwise(0)).alias("fraud_amount_total")
    )
    .orderBy("transaction_hour")
)

gold_fraud_by_time.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.fraud_by_time_of_day")
print("Created: gold.fraud_by_time_of_day")
display(gold_fraud_by_time)

Created: gold.fraud_by_time_of_day


time_of_day,transaction_hour,total_transactions,fraud_transactions,fraud_rate_pct,fraud_amount_total
Night,0,140582,13,0.009247272054743851,-622.78
Night,1,115586,38,0.032875953835239564,-1098.18
Night,2,112787,38,0.03369182618564197,1096.47
Night,3,103478,107,0.10340362202593788,14264.20
Night,4,114985,143,0.12436404748445451,3649.46
Night,5,182965,242,0.13226573388352963,15779.37
Morning,6,758856,455,0.05995867463655819,55242.34
Morning,7,901756,541,0.05999405604176739,57024.99
Morning,8,880501,530,0.06019300375581629,61682.53
Morning,9,876423,856,0.09766973253782704,97688.55


## 8. Average Transaction Amount: Fraud vs Non-Fraud

In [0]:
gold_fraud_vs_nonfrawd_amount = (df_trans
    .groupBy("is_fraud")
    .agg(
        count("*").alias("transaction_count"),
        avg("amount_clean").alias("avg_amount"),
        stddev("amount_clean").alias("stddev_amount"),
        min("amount_clean").alias("min_amount"),
        expr("percentile(amount_clean, 0.25)").alias("p25_amount"),
        expr("percentile(amount_clean, 0.50)").alias("median_amount"),
        expr("percentile(amount_clean, 0.75)").alias("p75_amount"),
        max("amount_clean").alias("max_amount"),
        sum("amount_clean").alias("total_amount")
    )
    .withColumn("fraud_status",
        when(col("is_fraud") == True, "Fraudulent")
        .when(col("is_fraud") == False, "Legitimate")
        .otherwise("Unknown")
    )
    .select("fraud_status", "transaction_count", "avg_amount", "median_amount", 
            "stddev_amount", "min_amount", "max_amount", "total_amount")
)

gold_fraud_vs_nonfrawd_amount.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.fraud_vs_nonfrawd_amount")
print("Created: gold.fraud_vs_nonfrawd_amount")
display(gold_fraud_vs_nonfrawd_amount)

Created: gold.fraud_vs_nonfrawd_amount


fraud_status,transaction_count,avg_amount,median_amount,stddev_amount,min_amount,max_amount,total_amount
Unknown,4390952,43.030150,28.99,81.91747707540384,-500.00,6820.20,188943324.74
Fraudulent,13332,110.234682,69.975,213.73620712760126,-500.00,4978.45,1469648.78
Legitimate,8901631,42.848614,28.95,81.12535009980606,-500.00,6613.44,381422548.76


## 9. Fraud by Device Type (Transaction Method)

In [0]:
gold_fraud_by_device = (df_trans
    .groupBy("transaction_method")
    .agg(
        count("*").alias("total_transactions"),
        sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
        (sum(when(col("is_fraud") == True, 1).otherwise(0)) / count("*") * 100).alias("fraud_rate_pct"),
        sum(when(col("is_fraud") == True, col("amount_clean")).otherwise(0)).alias("fraud_amount_total"),
        countDistinct(when(col("is_fraud") == True, col("client_id"))).alias("unique_fraud_users")
    )
    .orderBy(desc("fraud_rate_pct"))
)

gold_fraud_by_device.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.fraud_by_device_type")
print("Created: gold.fraud_by_device_type")
display(gold_fraud_by_device)

Created: gold.fraud_by_device_type


transaction_method,total_transactions,fraud_transactions,fraud_rate_pct,fraud_amount_total,unique_fraud_users
Online Transaction,1557912,8779,0.5635106475847159,1061612.39,1076
Chip Transaction,4780818,3176,0.06643214613064125,280756.35,591
Swipe Transaction,6967185,1377,0.019764079753874772,127280.04,557


## 10. Shared Device/Card Fraud (Multi-User Cards)

In [0]:
# Cards used by multiple users with fraud
gold_shared_card_fraud = (df_trans
    .filter(col("is_fraud") == True)
    .groupBy("card_id")
    .agg(
        countDistinct("client_id").alias("unique_users"),
        count("*").alias("fraud_transaction_count"),
        sum("amount_clean").alias("total_fraud_amount"),
        collect_set("client_id").alias("user_ids")
    )
    .filter(col("unique_users") > 1)  # Cards used by multiple users for fraud
    .join(df_cards, "card_id", "left")
    .select(
        "card_id",
        "card_brand",
        "card_type",
        "unique_users",
        "fraud_transaction_count",
        "total_fraud_amount",
        "card_on_dark_web_flag",
        "user_ids"
    )
    .orderBy(desc("unique_users"), desc("fraud_transaction_count"))
)

gold_shared_card_fraud.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.shared_device_fraud")
print("Created: gold.shared_device_fraud")
display(gold_shared_card_fraud.limit(20))


Created: gold.shared_device_fraud


card_id,card_brand,card_type,unique_users,fraud_transaction_count,total_fraud_amount,card_on_dark_web_flag,user_ids


## 11. Daily Fraud Losses

In [0]:
gold_daily_losses = (df_trans
    .filter(col("is_fraud") == True)
    .groupBy("transaction_date")
    .agg(
        count("*").alias("fraud_transaction_count"),
        sum("amount_clean").alias("total_fraud_loss"),
        avg("amount_clean").alias("avg_fraud_loss"),
        countDistinct("client_id").alias("unique_fraud_users"),
        countDistinct("merchant_id").alias("unique_fraud_merchants")
    )
    .orderBy("transaction_date")
    
    # Add cumulative loss
    .withColumn("cumulative_fraud_loss",
        sum("total_fraud_loss").over(
            Window.orderBy("transaction_date")
            .rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    )
)

gold_daily_losses.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.daily_fraud_losses")
print("✓ Created: gold.daily_fraud_losses")
display(gold_daily_losses.orderBy(desc("transaction_date")).limit(30))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ Created: gold.daily_fraud_losses


transaction_date,fraud_transaction_count,total_fraud_loss,avg_fraud_loss,unique_fraud_users,unique_fraud_merchants,cumulative_fraud_loss
2019-10-27,1,-210.00,-210.000000,1,1,1469648.78
2019-10-26,6,976.92,162.820000,5,5,1469858.78
2019-10-25,9,132.97,14.774444,8,8,1468881.86
2019-10-24,8,1165.86,145.732500,6,7,1468748.89
2019-10-22,11,1225.82,111.438182,8,10,1467583.03
2019-10-19,10,792.02,79.202000,8,9,1466357.21
2019-10-18,11,1165.11,105.919091,9,7,1465565.19
2019-10-17,11,1840.08,167.280000,9,10,1464400.08
2019-10-15,16,1286.36,80.397500,10,13,1462560.00
2019-10-12,13,651.62,50.124615,11,9,1461273.64


## 12. Weekly Unique Fraud Users

In [0]:
gold_weekly_fraud_users = (df_trans
    .filter(col("is_fraud") == True)
    .groupBy("transaction_year", "transaction_week")
    .agg(
        countDistinct("client_id").alias("unique_fraud_users"),
        count("*").alias("fraud_transaction_count"),
        sum("amount_clean").alias("total_fraud_amount"),
        countDistinct("merchant_id").alias("unique_merchants"),
        countDistinct("mcc_code").alias("unique_categories")
    )
    .orderBy("transaction_year", "transaction_week")
    
    # Add week-over-week growth
    .withColumn("prev_week_users",
        lag("unique_fraud_users", 1).over(
            Window.orderBy("transaction_year", "transaction_week")
        )
    )
    .withColumn("user_growth_pct",
        when(col("prev_week_users").isNotNull() & (col("prev_week_users") > 0),
             ((col("unique_fraud_users") - col("prev_week_users")) / col("prev_week_users") * 100)
        )
    )
)

gold_weekly_fraud_users.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.weekly_unique_fraud_users")
print("Created: gold.weekly_unique_fraud_users")
display(gold_weekly_fraud_users.orderBy(desc("transaction_year"), desc("transaction_week")).limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Created: gold.weekly_unique_fraud_users


transaction_year,transaction_week,unique_fraud_users,fraud_transaction_count,total_fraud_amount,unique_merchants,unique_categories,prev_week_users,user_growth_pct
2019,43,17,35,3291.57,22,20,22,-22.727272727272727
2019,42,22,48,5083.57,26,20,24,-8.333333333333332
2019,41,24,52,4196.77,27,18,18,33.33333333333333
2019,40,18,42,4854.91,22,18,13,38.46153846153847
2019,39,13,25,3431.37,18,16,10,30.0
2019,38,10,16,832.75,11,11,8,25.0
2019,37,8,13,981.60,11,9,12,-33.33333333333333
2019,36,12,36,3045.75,26,21,15,-20.0
2019,35,15,34,2978.36,20,17,11,36.36363636363637
2019,34,11,23,2462.98,20,17,15,-26.666666666666668


## 13. Fraud Seasonal Patterns

In [0]:
gold_fraud_seasonal = (df_trans
    .groupBy("transaction_year", "transaction_month")
    .agg(
        count("*").alias("total_transactions"),
        sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
        (sum(when(col("is_fraud") == True, 1).otherwise(0)) / count("*") * 100).alias("fraud_rate_pct"),
        sum(when(col("is_fraud") == True, col("amount_clean")).otherwise(0)).alias("fraud_amount_total"),
        countDistinct(when(col("is_fraud") == True, col("client_id"))).alias("unique_fraud_users")
    )
    .withColumn("month_name", 
        when(col("transaction_month") == 1, "January")
        .when(col("transaction_month") == 2, "February")
        .when(col("transaction_month") == 3, "March")
        .when(col("transaction_month") == 4, "April")
        .when(col("transaction_month") == 5, "May")
        .when(col("transaction_month") == 6, "June")
        .when(col("transaction_month") == 7, "July")
        .when(col("transaction_month") == 8, "August")
        .when(col("transaction_month") == 9, "September")
        .when(col("transaction_month") == 10, "October")
        .when(col("transaction_month") == 11, "November")
        .when(col("transaction_month") == 12, "December")
    )
    .orderBy("transaction_year", "transaction_month")
)

gold_fraud_seasonal.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.fraud_seasonal_patterns")
print("Created: gold.fraud_seasonal_patterns")
display(gold_fraud_seasonal)

Created: gold.fraud_seasonal_patterns


transaction_year,transaction_month,total_transactions,fraud_transactions,fraud_rate_pct,fraud_amount_total,unique_fraud_users,month_name
2010,1,101209,107,0.10572182315801955,12253.57,22,January
2010,2,93470,259,0.2770942548411255,32488.11,40,February
2010,3,103345,261,0.2525521312109923,37454.20,35,March
2010,4,100169,237,0.23660014575367627,29130.42,34,April
2010,5,104773,274,0.26151775743750777,27258.86,37,May
2010,6,102677,182,0.17725488668348316,15926.29,31,June
2010,7,106034,244,0.23011486881566293,28099.77,31,July
2010,8,107547,229,0.2129301607669205,24878.75,35,August
2010,9,103902,193,0.1857519585763508,22063.84,30,September
2010,10,106150,224,0.2110221384832784,29450.98,30,October


## 14. User Behavior Before vs After Frau

In [0]:
from pyspark.sql.functions import sum, min, count, avg, first, countDistinct, when, col
# Identify first fraud date for each user
first_fraud_dates = (df_trans
    .filter(col("is_fraud") == True)
    .groupBy("client_id")
    .agg(min("transaction_date").alias("first_fraud_date"))
)

# Calculate metrics before and after first fraud
user_behavior_change = (df_trans
    .join(first_fraud_dates, "client_id", "inner")
    .withColumn("period",
        when(col("transaction_date") < col("first_fraud_date"), "Before Fraud")
        .when(col("transaction_date") >= col("first_fraud_date"), "After Fraud")
    )
    .groupBy("client_id", "first_fraud_date", "period")
    .agg(
        count("*").alias("transaction_count"),
        avg("amount_clean").alias("avg_transaction_amount"),
        sum("amount_clean").alias("total_spent"),
        countDistinct("merchant_id").alias("unique_merchants"),
        countDistinct("mcc_code").alias("unique_categories"),
        countDistinct("transaction_method").alias("unique_methods")
    )
)

# Pivot to compare before vs after
# Use underscores instead of spaces to avoid Delta column name issues
gold_behavior_before_after = (user_behavior_change
    .groupBy("client_id", "first_fraud_date")
    .pivot("period", ["Before Fraud", "After Fraud"])
    .agg(
        first("transaction_count").alias("transactions"),
        first("avg_transaction_amount").alias("avg_amount"),
        first("total_spent").alias("total_spent"),
        first("unique_merchants").alias("merchants"),
        first("unique_categories").alias("categories")
    )
    # Rename columns to remove spaces
    .withColumnRenamed("Before Fraud_transactions", "before_fraud_transactions")
    .withColumnRenamed("After Fraud_transactions", "after_fraud_transactions")
    .withColumnRenamed("Before Fraud_avg_amount", "before_fraud_avg_amount")
    .withColumnRenamed("After Fraud_avg_amount", "after_fraud_avg_amount")
    .withColumnRenamed("Before Fraud_total_spent", "before_fraud_total_spent")
    .withColumnRenamed("After Fraud_total_spent", "after_fraud_total_spent")
    .withColumnRenamed("Before Fraud_merchants", "before_fraud_merchants")
    .withColumnRenamed("After Fraud_merchants", "after_fraud_merchants")
    .withColumnRenamed("Before Fraud_categories", "before_fraud_categories")
    .withColumnRenamed("After Fraud_categories", "after_fraud_categories")
    .join(df_users, col("client_id") == df_users.user_id, "left")
    .select(
        "client_id",
        df_users.credit_score,
        df_users.yearly_income_clean,
        "first_fraud_date",
        "before_fraud_transactions",
        "after_fraud_transactions",
        "before_fraud_avg_amount",
        "after_fraud_avg_amount",
        "before_fraud_total_spent",
        "after_fraud_total_spent"
    )
    .withColumn("transaction_count_change_pct",
        when(col("before_fraud_transactions") > 0,
             ((col("after_fraud_transactions") - col("before_fraud_transactions")) / col("before_fraud_transactions") * 100)
        )
    )
    .withColumn("avg_amount_change_pct",
        when(col("before_fraud_avg_amount") > 0,
             ((col("after_fraud_avg_amount") - col("before_fraud_avg_amount")) / col("before_fraud_avg_amount") * 100)
        )
    )
)

gold_behavior_before_after.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.user_behavior_before_after_fraud")
print("Created: gold.user_behavior_before_after_fraud")
display(gold_behavior_before_after.limit(20))


Created: gold.user_behavior_before_after_fraud


client_id,credit_score,yearly_income_clean,first_fraud_date,before_fraud_transactions,after_fraud_transactions,before_fraud_avg_amount,after_fraud_avg_amount,before_fraud_total_spent,after_fraud_total_spent,transaction_count_change_pct,avg_amount_change_pct
94,593,36760.00,2012-05-10,2975,10185,38.599916,35.181083,114834.75,358319.33,242.35294117647058,-8.8570995853980614880
1257,721,48259.00,2015-05-22,3118,2650,37.204448,38.593789,116003.47,102273.54,-15.0096215522771,3.7343411196424685564
1910,562,35868.00,2010-06-17,861,18161,30.855993,31.958193,26567.01,580392.74,2009.2915214866434,3.5720775539455171642
1983,773,141161.00,2016-12-11,8046,3322,52.701218,53.087246,424034.00,176355.83,-58.712403678846634,0.7324840196292996492
803,635,46487.00,2015-12-16,5586,5062,18.708246,28.323524,104504.26,143373.68,-9.380594343000357,51.3959352469493933317
569,662,45625.00,2019-01-29,1959,500,39.803890,43.580180,77975.82,21790.09,-74.47677386421644,9.4872385588443742559
1897,741,104049.00,2019-09-24,14156,151,48.513302,55.719603,686754.30,8413.66,-98.93331449562022,14.8542785234449718553
362,792,48710.00,2010-08-17,703,10438,71.035562,68.291783,49938.00,712829.63,1384.7795163584638,-3.8625428204537890472
638,679,16889.00,2012-06-21,1255,3529,84.192390,78.729362,105661.45,277835.92,181.19521912350598,-6.4887432225169044376
738,707,63974.00,2012-01-19,2257,8555,72.539570,69.404607,163721.81,593756.41,279.04297740363313,-4.3217281271449499907


## 15. Fraud by Transaction Value Ranges

In [0]:
gold_fraud_by_value = (df_trans
    .withColumn("amount_range",
        when(col("amount_clean") < 10, "< $10")
        .when((col("amount_clean") >= 10) & (col("amount_clean") < 50), "$10-$50")
        .when((col("amount_clean") >= 50) & (col("amount_clean") < 100), "$50-$100")
        .when((col("amount_clean") >= 100) & (col("amount_clean") < 250), "$100-$250")
        .when((col("amount_clean") >= 250) & (col("amount_clean") < 500), "$250-$500")
        .when((col("amount_clean") >= 500) & (col("amount_clean") < 1000), "$500-$1000")
        .otherwise("$1000+")
    )
    .groupBy("amount_range")
    .agg(
        count("*").alias("total_transactions"),
        sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
        (sum(when(col("is_fraud") == True, 1).otherwise(0)) / count("*") * 100).alias("fraud_rate_pct"),
        avg("amount_clean").alias("avg_transaction_amount"),
        sum(when(col("is_fraud") == True, col("amount_clean")).otherwise(0)).alias("fraud_amount_total")
    )
    # Order by custom sort
    .withColumn("sort_order",
        when(col("amount_range") == "< $10", 1)
        .when(col("amount_range") == "$10-$50", 2)
        .when(col("amount_range") == "$50-$100", 3)
        .when(col("amount_range") == "$100-$250", 4)
        .when(col("amount_range") == "$250-$500", 5)
        .when(col("amount_range") == "$500-$1000", 6)
        .otherwise(7)
    )
    .orderBy("sort_order")
    .drop("sort_order")
)

gold_fraud_by_value.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.fraud_by_transaction_value")
print("✓ Created: gold.fraud_by_transaction_value")
display(gold_fraud_by_value)

✓ Created: gold.fraud_by_transaction_value


amount_range,total_transactions,fraud_transactions,fraud_rate_pct,avg_transaction_amount,fraud_amount_total
< $10,3574928,2617,0.07320427152658739,-15.375969,-131678.23
$10-$50,5271899,2814,0.05337735036274405,26.826793,80459.99
$50-$100,2899476,2819,0.09722446400659981,70.759080,207830.10
$100-$250,1355171,3513,0.2592292780763461,139.613653,548185.25
$250-$500,161077,1213,0.7530559918548272,343.569585,424277.18
$500-$1000,34179,256,0.7489979227010737,695.260082,170587.62
$1000+,9185,100,1.0887316276537833,1296.274260,169986.87


## Summary: All Gold Tables Created

In [0]:
gold_tables = [
    "fraud_by_day_of_week",
    "fraud_rate_trend",
    "top_fraud_users",
    "user_spending_anomalies",
    "fraud_by_merchant_category",
    "fraud_by_merchant",
    "fraud_by_time_of_day",
    "fraud_vs_nonfrawd_amount",
    "fraud_by_device_type",
    "shared_device_fraud",
    "daily_fraud_losses",
    "weekly_unique_fraud_users",
    "fraud_seasonal_patterns",
    "user_behavior_before_after_fraud",
    "fraud_by_transaction_value"
]

print("=" * 80)
print("GOLD LAYER SUMMARY")
print("=" * 80)

for table in gold_tables:
    count = spark.table(f"{GOLD_PATH}.{table}").count()
    print(f"✓ {GOLD_PATH}.{table}: {count:,} rows")

print("\n" + "=" * 80)
print("Gold layer pipeline completed successfully!")
print("All analytics tables ready for dashboard creation!")
print("=" * 80)

GOLD LAYER SUMMARY
✓ databricks_workspace.gold.fraud_by_day_of_week: 7 rows
✓ databricks_workspace.gold.fraud_rate_trend: 3,591 rows
✓ databricks_workspace.gold.top_fraud_users: 1,196 rows
✓ databricks_workspace.gold.user_spending_anomalies: 0 rows
✓ databricks_workspace.gold.fraud_by_merchant_category: 109 rows
✓ databricks_workspace.gold.fraud_by_merchant: 87,827 rows
✓ databricks_workspace.gold.fraud_by_time_of_day: 24 rows
✓ databricks_workspace.gold.fraud_vs_nonfrawd_amount: 3 rows
✓ databricks_workspace.gold.fraud_by_device_type: 3 rows
✓ databricks_workspace.gold.shared_device_fraud: 0 rows
✓ databricks_workspace.gold.daily_fraud_losses: 1,571 rows
✓ databricks_workspace.gold.weekly_unique_fraud_users: 329 rows
✓ databricks_workspace.gold.fraud_seasonal_patterns: 118 rows
✓ databricks_workspace.gold.user_behavior_before_after_fraud: 1,196 rows
✓ databricks_workspace.gold.fraud_by_transaction_value: 7 rows

Gold layer pipeline completed successfully!
All analytics tables ready fo